# Valorant Player Rank Classification — Model Training

**Goal:** predict a player's rank (`rating` column: Radiant / Immortal 1 / 2 / 3 …) from their in-game stats.

What this notebook does:
1. Config
2. Load the data (Kaggle via `kagglehub`, or a local CSV)
3. Quick EDA
4. Clean the data
5. Train/test split (+ save `test_data.csv` for the Streamlit app)
6. Build a preprocessing pipeline
7. Train 5 models and compute 6 metrics
8. Save models + results to `model/`
9. Sanity check: reload a saved model and score `test_data.csv` (this is exactly what `app.py` will do)

In [14]:
import os, glob, json, time
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef)
import sklearn

# Note these versions: pin the SAME ones in requirements.txt, or the .pkl files may fail to load on Streamlit Cloud
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

scikit-learn 1.9.1 | pandas 3.1.1 | numpy 2.4.6


## 1. Config
All the knobs in one place. The two `MAX_*_ROWS` caps exist because **SVM and KNN get very slow** on 60k+ rows
(SVM training scales roughly with n² or worse). Set them to `None` to use everything.

In [15]:
RANDOM_STATE    = 42
TARGET_COL      = "rating"          # what we predict (player rank)
DROP_COLS       = ["name", "tag"]   # player identifiers -> no predictive meaning
TEST_SIZE       = 0.2
MAX_TRAIN_ROWS  = 20000             # stratified sample for training (None = all)
MAX_TEST_ROWS   = 5000              # keeps test_data.csv small for the Streamlit upload
MIN_CLASS_COUNT = 100               # drop ranks with too few players to learn from
MODEL_DIR       = "model"

os.makedirs(MODEL_DIR, exist_ok=True)

## 2. Load the data
Set `USE_KAGGLEHUB = False` if the Kaggle download doesn't work (e.g. no internet on the Virtual Lab) and put
`val_stats.csv` next to this notebook instead.

`keep_default_na=False` matters here: by default pandas reads the **region "NA" (North America) as missing**.

In [16]:
USE_KAGGLEHUB  = False
LOCAL_CSV_PATH = "val_stats.csv"

if USE_KAGGLEHUB:
    import kagglehub
    # Download latest version
    path = kagglehub.dataset_download("aliibrahim10/valorant-stats")
    print("Path to dataset files:", path)
    csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
    print("CSV files found:", csv_files)
    csv_path = csv_files[0]
else:
    csv_path = LOCAL_CSV_PATH

df = pd.read_csv(csv_path, low_memory=False,
                 keep_default_na=False, na_values=["", "NaN", "nan", "null"])
print("Shape:", df.shape)
df.head()

Shape: (85678, 38)


,region,name,tag,rating,damage_round,headshots,headshot_percent,aces,clutches,flawless,...,gun2_name,gun2_head,gun2_body,gun2_legs,gun2_kills,gun3_name,gun3_head,gun3_body,gun3_legs,gun3_kills
0,NA,ShimmyXD,#NA1,Radiant,135.8,992,24.9,0,140,80,...,Phantom,33,62,5,220,Classic,36,60,3,147
1,NA,XSET Cryo,#cells,Radiant,170.3,879,28.3,2,122,94,...,Operator,8,91,0,226,Phantom,32,63,5,137
2,NA,PuRelittleone,#yoruW,Radiant,147.5,720,24.0,3,117,59,...,Phantom,36,61,3,231,Operator,8,91,1,102
3,NA,Boba,#0068,Radiant,178.2,856,37.3,3,83,49,...,Sheriff,48,51,1,48,Phantom,44,56,0,36
4,NA,i love mina,#kelly,Radiant,149.8,534,24.4,2,71,38,...,Spectre,21,71,8,65,Operator,8,92,0,64


## 3. Quick EDA
Things to look for: column types (some numbers are stored as text like `"1,127"`), missing values,
duplicates, and **class balance** of the target.

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 85678 entries, 0 to 85677
Data columns (total 38 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   region            85678 non-null  str    
 1   name              85675 non-null  str    
 2   tag               85358 non-null  str    
 3   rating            85678 non-null  str    
 4   damage_round      85678 non-null  float64
 5   headshots         85678 non-null  str    
 6   headshot_percent  85678 non-null  float64
 7   aces              85678 non-null  int64  
 8   clutches          85678 non-null  int64  
 9   flawless          85678 non-null  int64  
 10  first_bloods      85678 non-null  str    
 11  kills             85678 non-null  str    
 12  deaths            85678 non-null  str    
 13  assists           85678 non-null  str    
 14  kd_ratio          85678 non-null  float64
 15  kills_round       85678 non-null  float64
 16  most_kills        85678 non-null  int64  
 17  scor

In [18]:
print("Missing values (top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10))
print("\nDuplicate rows:", df.duplicated().sum())
print(f"\nTarget distribution ({TARGET_COL}):")
print(df[TARGET_COL].value_counts())

Missing values (top 10):
agent_3             5217
agent_2             1987
tag                  320
name                   3
region                 0
rating                 0
headshots              0
damage_round           0
clutches               0
headshot_percent       0
dtype: int64

Duplicate rows: 1084

Target distribution (rating):
rating
Immortal 1    51315
Immortal 2    20716
Immortal 3    10936
Radiant        2607
Diamond 3        42
Platinum 1       10
Unrated           9
Gold 2            8
Gold 1            5
Gold 3            5
Silver 3          5
Silver 2          4
Bronze 2          4
Platinum 2        3
Diamond 1         3
Bronze 3          2
Silver 1          2
Diamond 2         1
Platinum 3        1
Name: count, dtype: int64


## 4. Clean the data
- Convert text-numbers like `"1,127"` → `1127`
- Drop identifier columns and duplicate rows
- Drop rows with no target, and ranks with fewer than `MIN_CLASS_COUNT` players

In [19]:
def clean_numeric_strings(data):
    """Turn object columns that are really numbers (e.g. '1,127') into numeric columns."""
    data = data.copy()
    for col in data.select_dtypes(exclude="number").columns:   # text columns
        converted = pd.to_numeric(
            data[col].astype(str).str.replace(",", "", regex=False).str.strip(),
            errors="coerce")
        non_null = data[col].notna().sum()
        # only convert if ~all values parse as numbers (so 'Vandal', 'Jett' etc. stay text)
        if non_null > 0 and converted.notna().sum() >= 0.95 * non_null:
            data[col] = converted
    return data

df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()
df = clean_numeric_strings(df.drop(columns=[TARGET_COL])).assign(**{TARGET_COL: df[TARGET_COL]})
df = df.drop_duplicates()
df = df[df[TARGET_COL].ne("") & df[TARGET_COL].ne("nan")]

counts = df[TARGET_COL].value_counts()
df = df[df[TARGET_COL].isin(counts[counts >= MIN_CLASS_COUNT].index)]

print("Shape after cleaning:", df.shape)
print(df[TARGET_COL].value_counts())

Shape after cleaning: (84492, 36)
rating
Immortal 1    50520
Immortal 2    20530
Immortal 3    10858
Radiant        2584
Name: count, dtype: int64


In [20]:
X = df.drop(columns=[TARGET_COL])
y_raw = df[TARGET_COL]

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
print(f"Features: {X.shape[1]}  (numeric: {len(num_cols)}, categorical: {len(cat_cols)})")
print("Categorical:", cat_cols)

# Ranks are text -> encode as 0..k-1 (models need numbers). Saved so the app can decode.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = label_encoder.classes_.tolist()
print("Classes:", class_names)

Features: 35  (numeric: 28, categorical: 7)
Categorical: ['region', 'agent_1', 'agent_2', 'agent_3', 'gun1_name', 'gun2_name', 'gun3_name']
Classes: ['Immortal 1', 'Immortal 2', 'Immortal 3', 'Radiant']


## 5. Train/test split
`stratify=y` keeps the rank proportions the same in train and test. The test split is saved as
**`test_data.csv`** — this is the file you upload in the Streamlit app.

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

if MAX_TRAIN_ROWS and len(X_train) > MAX_TRAIN_ROWS:
    X_train, _, y_train, _ = train_test_split(
        X_train, y_train, train_size=MAX_TRAIN_ROWS, stratify=y_train, random_state=RANDOM_STATE)
if MAX_TEST_ROWS and len(X_test) > MAX_TEST_ROWS:
    X_test, _, y_test, _ = train_test_split(
        X_test, y_test, train_size=MAX_TEST_ROWS, stratify=y_test, random_state=RANDOM_STATE)

print("Train:", X_train.shape, " Test:", X_test.shape)

test_df = X_test.copy()
test_df[TARGET_COL] = label_encoder.inverse_transform(y_test)
test_df.to_csv("test_data.csv", index=False)
print("Saved test_data.csv")

Train: (20000, 35)  Test: (5000, 35)
Saved test_data.csv


## 6. Preprocessing pipeline
- **Numeric:** fill missing with median → standardise (needed for LR, KNN, SVM; harmless for trees)
- **Categorical** (region, agents, guns): fill missing with most frequent → one-hot encode

Each model is wrapped in a `Pipeline(preprocess → model)`, so **the saved `.pkl` contains the scaler/encoder too**.
The app can then feed raw test rows straight in, with no separate scaler file to keep in sync.

In [22]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
])

## 7. Train the 5 models and evaluate
Multi-class notes:
- **AUC** uses predicted probabilities, one-vs-rest, weighted by class size
- **Precision / Recall / F1** use `average="weighted"` (binary would use the default)
- **MCC** works for multi-class directly

In [23]:
models = {
    "Logistic Regression":    LogisticRegression(max_iter=2000),
    "Decision Tree":          DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
    "KNN":                    KNeighborsClassifier(n_neighbors=7),
    "Naive Bayes":            GaussianNB(),
    "Support Vector Machine": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),  # probability=True -> needed for AUC
}

def evaluate(y_true, y_pred, y_proba, n_classes):
    avg = "binary" if n_classes == 2 else "weighted"
    if n_classes == 2:
        auc = roc_auc_score(y_true, y_proba[:, 1])
    else:
        auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted",
                            labels=list(range(n_classes)))
    return {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "AUC":       auc,
        "Precision": precision_score(y_true, y_pred, average=avg, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, average=avg, zero_division=0),
        "F1":        f1_score(y_true, y_pred, average=avg, zero_division=0),
        "MCC":       matthews_corrcoef(y_true, y_pred),
    }

In [24]:
results = {}
for name, clf in models.items():
    start = time.time()
    pipe = Pipeline([("preprocess", preprocessor), ("model", clf)])
    pipe.fit(X_train, y_train)

    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)
    results[name] = evaluate(y_test, y_pred, y_proba, len(class_names))

    file_name = name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(pipe, os.path.join(MODEL_DIR, file_name), compress=3)
    print(f"{name:<24} done in {time.time() - start:6.1f}s  -> {MODEL_DIR}/{file_name}")

results_df = pd.DataFrame(results).T.round(4)
results_df

Logistic Regression      done in    4.1s  -> model/logistic_regression.pkl
Decision Tree            done in    0.7s  -> model/decision_tree.pkl
KNN                      done in    1.3s  -> model/knn.pkl
Naive Bayes              done in    0.4s  -> model/naive_bayes.pkl


c:\Users\shubh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Support Vector Machine   done in  797.8s  -> model/support_vector_machine.pkl


,Accuracy,AUC,Precision,Recall,F1,MCC
Logistic Regression,0.6106,0.7417,0.5369,0.6106,0.5444,0.2053
Decision Tree,0.5792,0.7012,0.5329,0.5792,0.5454,0.1757
KNN,0.5724,0.6615,0.5069,0.5724,0.5223,0.1309
Naive Bayes,0.0544,0.5897,0.5020,0.0544,0.0466,0.0161
Support Vector Machine,0.6058,0.7271,0.4871,0.6058,0.4921,0.1427


## 8. Save results and metadata
- `model/metrics.csv` — the comparison table
- `model/label_encoder.pkl` — turns 0..k-1 back into rank names in the app
- `model/metadata.json` — target column, feature list and class names, so the app can validate an uploaded CSV

The printed markdown table can be pasted straight into the README.

In [25]:
results_df.to_csv(os.path.join(MODEL_DIR, "metrics.csv"))
joblib.dump(label_encoder, os.path.join(MODEL_DIR, "label_encoder.pkl"))
with open(os.path.join(MODEL_DIR, "metadata.json"), "w") as f:
    json.dump({"target_col": TARGET_COL, "feature_cols": X.columns.tolist(),
               "num_cols": num_cols, "cat_cols": cat_cols, "class_names": class_names}, f, indent=2)

# Markdown table for the README
cols = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
lines = ["| ML Model Name | " + " | ".join(cols) + " |", "|" + "---|" * (len(cols) + 1)]
for name, row in results_df.iterrows():
    lines.append(f"| {name} | " + " | ".join(f"{row[c]:.4f}" for c in cols) + " |")
print("\n".join(lines))

| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
|---|---|---|---|---|---|---|
| Logistic Regression | 0.6106 | 0.7417 | 0.5369 | 0.6106 | 0.5444 | 0.2053 |
| Decision Tree | 0.5792 | 0.7012 | 0.5329 | 0.5792 | 0.5454 | 0.1757 |
| KNN | 0.5724 | 0.6615 | 0.5069 | 0.5724 | 0.5223 | 0.1309 |
| Naive Bayes | 0.0544 | 0.5897 | 0.5020 | 0.0544 | 0.0466 | 0.0161 |
| Support Vector Machine | 0.6058 | 0.7271 | 0.4871 | 0.6058 | 0.4921 | 0.1427 |


## 9. Sanity check — score `test_data.csv` like the app will
Load a saved model from disk, read the test CSV, predict. **No retraining.** If this works, `app.py` will too.

In [26]:
check_model = joblib.load(os.path.join(MODEL_DIR, "logistic_regression.pkl"))
le          = joblib.load(os.path.join(MODEL_DIR, "label_encoder.pkl"))
meta        = json.load(open(os.path.join(MODEL_DIR, "metadata.json")))

uploaded = pd.read_csv("test_data.csv", keep_default_na=False, na_values=["", "NaN", "nan", "null"])
X_up = uploaded[meta["feature_cols"]]
y_up = le.transform(uploaded[meta["target_col"]].astype(str))
print("Accuracy from reloaded model:", round(accuracy_score(y_up, check_model.predict(X_up)), 4))

Accuracy from reloaded model: 0.6106
